# Intertidal ? barnacle scientist workbench

Run your trained PointRend model inside Colab. Upload a photograph, inspect zoomable masks, select individual observations, and export pixel measurements. Choose **Runtime ? Change runtime type ? T4 GPU**.

Use your existing trained checkpoint; this notebook does not retrain. Download and upload **barnacle_scientist_demo.zip** in the first cell. Your original training notebook and checkpoints remain available.


In [ ]:
from pathlib import Path
import io, sys, zipfile
from google.colab import files, drive

drive.mount('/content/drive')
WORKSPACE = Path(globals().get('WORKSPACE', '/content/barnacle_scientist_workspace'))
WORKSPACE.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # Select barnacle_scientist_demo.zip
names = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(names) == 1, 'Choose exactly one scientist demo ZIP.'
with zipfile.ZipFile(io.BytesIO(uploaded[names[0]])) as archive:
    for member in archive.infolist():
        relative = Path(member.filename)
        assert relative.parts and relative.parts[0] in {'training','src','demo','notebooks','docs'}
        target = (WORKSPACE / relative).resolve()
        assert target.is_relative_to(WORKSPACE.resolve()), 'Unsafe ZIP path'
        assert (member.external_attr >> 16) & 0o170000 != 0o120000, 'Symlinks are unsupported'
        if not member.is_dir():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
del uploaded
print('Workbench files:', WORKSPACE)


## Runtime setup

An existing working Detectron2 session is reused. On a fresh runtime this uses the fixed installer from the training pipeline. Installation can take several minutes. A runtime restart may be needed if you previously imported an incompatible installation.


In [ ]:
import importlib.util, subprocess, torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime and reconnect.'
if importlib.util.find_spec('detectron2') is None:
    subprocess.run([sys.executable, str(WORKSPACE / 'training/colab_setup.py'),
                    '--checkout', '/content/detectron2', '--revision', 'main'], check=True)
from detectron2.projects.point_rend import add_pointrend_config
from PIL import Image
print('Ready:', torch.cuda.get_device_name())


## Select the checkpoint

If continuing in the training session, `LATEST_OUTPUT` is used. Otherwise set `TRAINED_RUN` to the exact Drive folder containing your desired **model_final.pth** and **config.yaml**. The cell lists available final checkpoints when no run is selected. It does not silently choose an older model.


In [ ]:
TRAINED_RUN = globals().get('LATEST_OUTPUT')
# For a fresh session, replace the line above with your actual path, for example:
# TRAINED_RUN = Path('/content/drive/MyDrive/BarnacleChallenge/runs/YOUR_RUN/finetune_to_400_to_600')
CHECKPOINT = 'model_final.pth'
if TRAINED_RUN is None:
    available = sorted(Path('/content/drive/MyDrive/BarnacleChallenge/runs').rglob('model_final.pth'))
    for path in available:
        print(path.parent)
    raise ValueError('Set TRAINED_RUN above to your chosen run directory, then rerun this cell.')
TRAINED_RUN = Path(TRAINED_RUN)
assert (TRAINED_RUN / CHECKPOINT).is_file(), 'Checkpoint not found in the selected run.'
assert (TRAINED_RUN / 'config.yaml').is_file(), 'Matching config.yaml is required.'
print('Selected checkpoint:', TRAINED_RUN / CHECKPOINT)


## Open the workbench

The first analysis loads the model once. Keep the runtime connected while using the UI. Use **Inside the green frame** for the first unseen photo, **Whole image** for the second. Both sample buttons select the appropriate mode automatically.

Measurements describe the learned annotation region in pixels. They do not establish whole-shell area or accuracy. The selected analysis region is drawn on the result for inspection.


In [ ]:
sys.path.insert(0, str(WORKSPACE / 'training'))
from scientist_colab import launch_scientist_demo
# Run once per session; reuse this UI for subsequent images.
demo = launch_scientist_demo(WORKSPACE, TRAINED_RUN, checkpoint=CHECKPOINT)


## Download the complete last analysis

CSV, JSON and overlay downloads are also available in the UI. The complete ZIP includes a standalone interactive `report.html`, lossless source image, masks, measurements and audit records. Output folders are saved beside your training run on Drive. Open the HTML locally to view results without a running model.


In [ ]:
if demo.last_output is not None:
    files.download(str(demo.last_output.with_suffix('.zip')))
else:
    print('Analyze a photograph first, then run this cell.')
